# 📐 Módulo 00 (Parte 2): Álgebra Lineal Geométrica & Cálculo en Grafos (DAGs)
## De las Proyecciones y SVD a la Regla de la Cadena como Recorrido Topológico

> *"El gradiente no es magia: es un mensaje que viaja hacia atrás a través de las aristas de un Grafo Dirigido Acíclico (DAG) en orden topológico inverso."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-usuario/algo-to-ai/blob/main/notebooks/00_foundations/01_linear_algebra_and_dag_calculus.ipynb)

---

### ⚙️ Inicialización del Entorno
Verificamos que las librerías matemáticas y de visualización estén cargadas.

In [ ]:
# !pip install -q numpy matplotlib torch
import math
import time
from typing import List, Tuple, Dict, Set
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fijar semillas para reproducibilidad determinista
np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo: NumPy, Matplotlib y PyTorch disponibles")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El misterio de las matrices: de resolver ecuaciones a transformar espacios
Durante más de un siglo, el álgebra matricial fue enseñada como una técnica mecánica para resolver sistemas lineales ($A x = b$). Sin embargo, a finales del siglo XIX, matemáticos como **Eugenio Beltrami (1873)** y **Camille Jordan (1874)** descubrieron que cualquier matriz rectangular puede descomponerse en rotaciones y estiramientos independientes a lo largo de ejes ortogonales canónicos: la **Descomposición en Valores Singulares (SVD)**.

En 1936, **Carl Eckart y Gale Young** demostraron el famoso *Teorema de Eckart-Young-Mirsky*: si quieres aproximar una matriz densa $W \in \mathbb{R}^{m \times n}$ mediante otra matriz de menor rango $r \ll \min(m, n)$, la solución óptima absoluta (que minimiza el error cuadrático de Frobenius) se obtiene reteniendo únicamente los $r$ valores singulares mayores. 

85 años después, esta misma propiedad matemática se convirtió en el cimiento de **LoRA (Low-Rank Adaptation, 2021)** para hacer fine-tuning de modelos masivos como LLaMA o GPT sin gastar millones de dólares.

### El Teorema de Baur & Strassen (1983): El milagro computacional del Deep Learning
Imagina que tienes una función con $N = 1.000.000$ parámetros y quieres calcular su gradiente (la derivada respecto a cada uno de los parámetros):
* **El método ingenuo (Diferencias Finitas):** Para cada parámetro $w_i$, evalúas $\frac{f(w_i + \epsilon) - f(w_i)}{\epsilon}$. Esto requiere $N + 1$ evaluaciones completas de la red. Si evaluar la red toma 1 segundo, calcular el gradiente tomaría **11.5 días**.
* **El descubrimiento de Baur & Strassen (1983):** Walter Baur y Volker Strassen demostraron formalmente que si una función escalar se computa mediante un Grafo Dirigido Acíclico (DAG) de operaciones aritméticas básicas, **calcular el gradiente completo de todas las $N$ variables cuesta a lo sumo un múltiplo constante (entre 3 y 5 veces) del tiempo de calcular la función original**, ¡independientemente de cuántas variables haya!

Este resultado teórico es la razón fundamental por la cual hoy podemos entrenar redes con 70.000 millones de parámetros en tiempo finito.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### A. El Producto Escalar como Filtro de Similitud y Proyección
En programación clásica, solemos buscar elementos mediante claves en tablas hash ($O(1)$) o búsquedas binarias. En Deep Learning, la búsqueda es **diferenciable y geométrica**, mediada por el producto escalar:

$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^d u_i v_i = \|\mathbf{u}\| \|\mathbf{v}\| \cos(\theta)$$

* Si $\theta = 0^\circ$ (apuntan en la misma dirección): el producto escalar es máximo positivo.
* Si $\theta = 90^\circ$ (ortogonales, independientes): el producto escalar es $0$.
* Si $\theta = 180^\circ$ (opuestos): el producto escalar es máximo negativo.

> **Conexión con Transformers:** La fórmula de atención $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$ no es más que una base de datos donde la consulta $Q$ y las llaves $K$ calculan su similitud mediante productos escalares continuos.

In [ ]:
# Visualización de la proyección y producto escalar
fig, ax = plt.subplots(figsize=(6, 6))

u = np.array([3.0, 1.0])
v = np.array([2.0, 3.0])

# Proyección ortogonal de v sobre u
proj_v_on_u = (np.dot(v, u) / np.dot(u, u)) * u

ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color='blue', label='Vector u (Referencia)')
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='green', label='Vector v')
ax.quiver(0, 0, proj_v_on_u[0], proj_v_on_u[1], angles='xy', scale_units='xy', scale=1, color='red', alpha=0.7, label='Proyección de v sobre u')

# Línea discontinua ortogonal
ax.plot([v[0], proj_v_on_u[0]], [v[1], proj_v_on_u[1]], 'k--', alpha=0.5)

ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)
ax.axhline(0, color='grey', alpha=0.3)
ax.axvline(0, color='grey', alpha=0.3)
ax.set_aspect('equal')
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend()
ax.set_title(f"Geometría del Producto Escalar\nu · v = {np.dot(u, v):.1f} | Proyección = ||v|| cos(θ)")
plt.show()

### B. Matrices como Transformaciones del Espacio
Una matriz $2 \times 2$ no es una tabla estática; es una función que distorsiona el plano cartesiano continuo.
Las columnas de la matriz definen exactamente **a dónde van a parar los vectores base canónicos** $\hat{i} = [1, 0]^T$ y $\hat{j} = [0, 1]^T$ tras la transformación.

In [ ]:
def visualizar_transformacion_lineal(A: np.ndarray, titulo: str = "Transformación Lineal 2D"):
    # Generar una cuadrícula de coordenadas original
    x = np.linspace(-2, 2, 9)
    y = np.linspace(-2, 2, 9)
    
    fig, (ax_orig, ax_trans) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Dibujar rejilla original
    for xi in x:
        ax_orig.plot([xi, xi], [-2, 2], color='lightgray')
        ax_trans.plot([A[0,0]*xi + A[0,1]*(-2), A[0,0]*xi + A[0,1]*2],
                      [A[1,0]*xi + A[1,1]*(-2), A[1,0]*xi + A[1,1]*2], color='lightgray')
    for yi in y:
        ax_orig.plot([-2, 2], [yi, yi], color='lightgray')
        ax_trans.plot([A[0,0]*(-2) + A[0,1]*yi, A[0,0]*2 + A[0,1]*yi],
                      [A[1,0]*(-2) + A[1,1]*yi, A[1,0]*2 + A[1,1]*yi], color='lightgray')
        
    # Vectores base originales
    ax_orig.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='red', label='i = [1, 0]')
    ax_orig.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='blue', label='j = [0, 1]')
    ax_orig.set_title("Espacio Original (Base Canónica)")
    ax_orig.set_xlim(-3, 3)
    ax_orig.set_ylim(-3, 3)
    ax_orig.set_aspect('equal')
    ax_orig.grid(True, alpha=0.3)
    ax_orig.legend()
    
    # Vectores base transformados
    trans_i = A @ np.array([1, 0])
    trans_j = A @ np.array([0, 1])
    ax_trans.quiver(0, 0, trans_i[0], trans_i[1], angles='xy', scale_units='xy', scale=1, color='red', label=f'A·i = {trans_i}')
    ax_trans.quiver(0, 0, trans_j[0], trans_j[1], angles='xy', scale_units='xy', scale=1, color='blue', label=f'A·j = {trans_j}')
    ax_trans.set_title(f"{titulo}\nMatriz A = {A.tolist()}")
    ax_trans.set_xlim(-3, 3)
    ax_trans.set_ylim(-3, 3)
    ax_trans.set_aspect('equal')
    ax_trans.grid(True, alpha=0.3)
    ax_trans.legend()
    
    plt.tight_layout()
    plt.show()

# Ejemplo: Matriz que rota 45 grados y escala
theta = np.pi / 4
A = np.array([
    [1.5 * np.cos(theta), -1.0 * np.sin(theta)],
    [1.5 * np.sin(theta),  1.0 * np.cos(theta)]
])
visualizar_transformacion_lineal(np.round(A, 2), "Rotación y Estiramiento")

### C. El Grafo Computacional (DAG) y la Regla de la Cadena Multivariable
Para un programador, el cálculo en Deep Learning no es una serie de ecuaciones simbólicas en papel. Es un **recorrido de grafos**:

1. **Forward Pass:** Ordenación topológica del DAG para computar valores desde las entradas hasta la pérdida escalar $\mathcal{L}$.
2. **Backward Pass:** Recorrido en orden topológico inverso (*Reverse Topological Sort*).

Por la regla de la cadena multivariable, si una variable $x$ alimenta a varios nodos intermedios $y_1, y_2, \dots, y_k$, el gradiente acumulado en $x$ es simplemente la suma ponderada por los caminos de salida:

$$\frac{\partial \mathcal{L}}{\partial x} = \sum_{y \in \text{hijos}(x)} \frac{\partial \mathcal{L}}{\partial y} \cdot \frac{\partial y}{\partial x}$$

Cada nodo únicamente necesita conocer dos cosas:
* Su gradiente entrante desde arriba (upstream gradient): $\frac{\partial \mathcal{L}}{\partial \text{nodo}}$.
* Su derivada local respecto a sus entradas directas: $\frac{\partial \text{nodo}}{\partial \text{padre}}$.

¡Multiplica ambos y envía el resultado hacia atrás a sus padres!

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

### Componente 1: Compresión de Matrices mediante Aproximación de Bajo Rango (SVD)
Demostremos el teorema de Eckart-Young-Mirsky construyendo un compresor de bajo rango con NumPy puro:

In [ ]:
def comprimir_matriz_svd(W: np.ndarray, rank_r: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Descompone W (m x n) en W_approx = B @ A donde:
    B = U[:, :r] @ diag(S[:r])  (m x r)
    A = Vh[:r, :]              (r x n)
    Retorna B, A, la matriz reconstruida y el ratio de compresión.
    """
    m, n = W.shape
    U, S, Vh = np.linalg.svd(W, full_matrices=False)
    
    # Retener solo los r valores singulares mayores
    U_r = U[:, :rank_r]
    S_r = S[:rank_r]
    Vh_r = Vh[:rank_r, :]
    
    # Factorización en dos matrices de bajo rango (estilo LoRA: B y A)
    B = U_r * S_r[np.newaxis, :]  # (m x r)
    A = Vh_r                      # (r x n)
    
    W_approx = B @ A
    
    params_originales = m * n
    params_comprimidos = (m * rank_r) + (rank_r * n)
    ratio_compresion = params_originales / params_comprimidos
    error_frobenius = np.linalg.norm(W - W_approx, 'fro') / np.linalg.norm(W, 'fro')
    
    return B, A, W_approx, ratio_compresion, error_frobenius

# Probemos con una matriz de pesos simulada de 200 x 200 con rango intrínseco concentrado
np.random.seed(42)
W_sintetica = (np.random.randn(200, 10) @ np.random.randn(10, 200)) + 0.05 * np.random.randn(200, 200)

r = 10
B, A, W_rec, ratio, err = comprimir_matriz_svd(W_sintetica, rank_r=r)
print(f"Matriz original:     {W_sintetica.shape} ({W_sintetica.size} parámetros)")
print(f"Matrices B y A (r={r}): B={B.shape}, A={A.shape} ({B.size + A.size} parámetros)")
print(f"⚡ Factor de compresión de memoria: {ratio:.2f}x")
print(f"📉 Error relativo de reconstrucción: {err * 100:.2f}%")

### Componente 2: Mini-DAG con Ordenación Topológica y Backward Pass
Vamos a construir un evaluador de grafos dirigido acíclico en Python puro para calcular derivadas exactas sin usar PyTorch:

In [ ]:
class DAGNode:
    """
    Nodo de un grafo computacional con diferenciación automática en reversa.
    """
    def __init__(self, value: float, children: Tuple['DAGNode', ...] = (), op: str = ""):
        self.value = float(value)
        self.grad = 0.0
        self.children = set(children)
        self.op = op
        self._backward = lambda: None

    def __add__(self, other: Union['DAGNode', float]) -> 'DAGNode':
        other = other if isinstance(other, DAGNode) else DAGNode(other)
        out = DAGNode(self.value + other.value, (self, other), "+")
        
        def _backward():
            # d(x + y)/dx = 1, d(x + y)/dy = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other: Union['DAGNode', float]) -> 'DAGNode':
        other = other if isinstance(other, DAGNode) else DAGNode(other)
        out = DAGNode(self.value * other.value, (self, other), "*")
        
        def _backward():
            # d(x * y)/dx = y, d(x * y)/dy = x
            self.grad += other.value * out.grad
            other.grad += self.value * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power: float) -> 'DAGNode':
        assert isinstance(power, (int, float)), "Solo potencias escalares soportadas"
        out = DAGNode(self.value ** power, (self,), f"**{power}")
        
        def _backward():
            # d(x^p)/dx = p * x^(p-1)
            self.grad += (power * (self.value ** (power - 1))) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        """
        Ejecuta el recorrido en ordenación topológica inversa (Reverse Topological Sort).
        """
        topo: List['DAGNode'] = []
        visited: Set['DAGNode'] = set()

        def build_topo(v: 'DAGNode'):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        
        # El gradiente de la salida respecto a sí misma es 1.0 (dL/dL = 1)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

print("✅ DAGNode from scratch compilado correctamente")

### Prueba del DAG: Calculando $L = (x_1 \cdot x_2 + x_3)^2$

In [ ]:
# Definir variables de entrada
x1 = DAGNode(2.0)
x2 = DAGNode(-3.0)
x3 = DAGNode(10.0)

# Grafo: L = (x1 * x2 + x3)**2 = (2.0 * -3.0 + 10.0)**2 = 4.0**2 = 16.0
prod = x1 * x2
suma = prod + x3
L = suma ** 2

print(f"Forward pass completado: L = {L.value} (Esperado: 16.0)")

# Ejecutar backward pass con ordenación topológica
L.backward()

print(f"dL/dx1 = {x1.grad} (Analítico: 2 * (x1*x2 + x3) * x2 = 2 * 4 * -3 = -24.0)")
print(f"dL/dx2 = {x2.grad} (Analítico: 2 * (x1*x2 + x3) * x1 = 2 * 4 * 2 = 16.0)")
print(f"dL/dx3 = {x3.grad} (Analítico: 2 * (x1*x2 + x3) * 1  = 2 * 4 * 1 = 8.0)")

---

## 4. ⚡ Transición a PyTorch Moderno

### Inspeccionando el Grafo Dinámico de PyTorch
En PyTorch, cada tensor con `requires_grad=True` mantiene un puntero hacia el nodo de operación que lo creó: `grad_fn`. A su vez, `grad_fn` tiene un atributo `next_functions` que apunta a sus nodos progenitores, formando el DAG dinámico.

In [ ]:
# Mismo cálculo en PyTorch
tx1 = torch.tensor(2.0, requires_grad=True)
tx2 = torch.tensor(-3.0, requires_grad=True)
tx3 = torch.tensor(10.0, requires_grad=True)

t_prod = tx1 * tx2
t_suma = t_prod + tx3
t_L = t_suma ** 2

print("Nodo raíz L.grad_fn:", t_L.grad_fn)
print("Padres directos de L:", t_L.grad_fn.next_functions)
print("Padres de la suma:   ", t_suma.grad_fn.next_functions)

# Ejecutar backward
t_L.backward()
print("\n--- Gradientes en PyTorch ---")
print("tx1.grad:", tx1.grad.item())
print("tx2.grad:", tx2.grad.item())
print("tx3.grad:", tx3.grad.item())

### Demostración Práctica: Capa LoRA Sintética en PyTorch
Veamos cómo se expresa en PyTorch la descomposición de bajo rango $W_0 + \frac{\alpha}{r}(B \cdot A)$:

In [ ]:
class MiniLoRALayer(torch.nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 16.0):
        super().__init__()
        # Peso original 'congelado' (no computa gradientes para ahorrar memoria)
        self.weight_frozen = torch.nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
        
        # Adaptadores de bajo rango entrenables
        # Matriz A inicializada aleatoriamente normalizada
        self.lora_A = torch.nn.Parameter(torch.randn(rank, in_features) / math.sqrt(rank))
        # Matriz B inicializada a ceros (para que al inicio Delta W = 0 exactamente)
        self.lora_B = torch.nn.Parameter(torch.zeros(out_features, rank))
        
        self.scale = alpha / rank
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Salida base congelada
        base_out = x @ self.weight_frozen.t()
        # Desvío de bajo rango
        lora_out = (x @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + self.scale * lora_out

layer = MiniLoRALayer(in_features=128, out_features=128, rank=4)
params_entrenables = sum(p.numel() for p in layer.parameters() if p.requires_grad)
params_totales = sum(p.numel() for p in layer.parameters())

print(f"Parámetros totales de la capa:      {params_totales}")
print(f"Parámetros entrenables con LoRA:    {params_entrenables}")
print(f"⚡ Reducción de parámetros a entrenar: {params_totales / params_entrenables:.1f}x veces menos")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Verificación de Baur-Strassen (Benchmark Empírico)
Vamos a comprobar el teorema de Baur-Strassen en la práctica: medir el tiempo de calcular el gradiente de una función cuadrática de $N$ variables usando:
1. Diferencias finitas numéricas ($N$ pasos hacia adelante).
2. PyTorch Autograd (1 forward + 1 backward por el grafo).

In [ ]:
N = 2000
x_vec = torch.randn(N, requires_grad=True)
A_mat = torch.randn(N, N)

# 1. Tiempo con PyTorch Autograd (Reverse-mode DAG)
t0 = time.perf_counter()
loss = torch.sum((x_vec @ A_mat) ** 2)
loss.backward()
t_autograd = time.perf_counter() - t0

# 2. Estimación con Diferencias Finitas para N variables
# En lugar de esperar minutos, medimos 10 variables y extrapolamos a N=2000
x_num = x_vec.detach().clone()
eps = 1e-4
t0 = time.perf_counter()
for k in range(10):
    x_plus = x_num.clone()
    x_plus[k] += eps
    loss_plus = torch.sum((x_plus @ A_mat) ** 2)
    grad_k = (loss_plus - loss.item()) / eps
t_10_samples = time.perf_counter() - t0
t_extrapolado = (t_10_samples / 10) * N

print(f"Tiempo Autograd (DAG Backward):         {t_autograd * 1000:.2f} ms")
print(f"Tiempo Estimado Diferencias Finitas:   {t_extrapolado * 1000:.2f} ms ({t_extrapolado:.2f} s)")
print(f"🚀 Aceleración del algoritmo de grafo: {t_extrapolado / t_autograd:.1f}x más rápido")

### Reto 2 (Para resolver): Agregar el Operador `sin` o `relu` a `DAGNode`
Extiende la clase `DAGNode` agregando una función no lineal:
1. Define el método `relu(self) -> DAGNode` o `sin(self) -> DAGNode`.
2. Implementa su derivada local dentro del closure `_backward()`.
3. Conecta el nuevo nodo en un grafo y verifica que su gradiente coincida con la aproximación numérica por diferencias finitas.

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def relu_node(node: DAGNode) -> DAGNode:
    # 1. Calcular el valor de salida: max(0, node.value)
    # 2. Definir la derivada local: out.grad * (1.0 si node.value > 0 else 0.0)
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Baur, W., & Strassen, V. (1983):** *"The Complexity of Partial Derivatives"*, Theoretical Computer Science, 22(3), 317-330. [ScienceDirect Link](https://doi.org/10.1016/0304-3975(83)90110-X)
   * *¿Qué leer?* Teorema 1 y la demostración de la cota $c \le 5$ para la evaluación de todas las derivadas parciales en grafos de complejidad computacional.
2. **Hu, E. J., Shen, Y., Wallis, P., et al. (2021):** *"LoRA: Low-Rank Adaptation of Large Language Models"*, arXiv:2106.09685. [arXiv Link](https://arxiv.org/abs/2106.09685)
   * *¿Qué leer?* Sección 4 ("Our Method"): la hipótesis del rango intrínseco y por qué congelar $W_0$ y aprender solo $B \cdot A$ retiene casi el 100% de la capacidad de adaptación.
3. **Eckart, C., & Young, G. (1936):** *"The approximation of one matrix by another of lower rank"*, Psychometrika, 1(3), 211-218. [Springer Link](https://doi.org/10.1007/BF02288367)
   * *¿Qué leer?* La formulación original de por qué los valores singulares minimizan el error de reconstrucción en norma $L_2$.

### 🔗 Libros y Cursos Recomendados
* **Gilbert Strang (MIT):** *Linear Algebra and Learning from Data* (2019) - Especialmente el capítulo sobre Deep Learning y SVD.
* **Griewank, A., & Walther, A.:** *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation* (SIAM, 2008) - La biblia académica del cálculo automático.